In [ ]:
import os
import pandas as pd
import boto3
import kagglehub
from tqdm import tqdm

## Helper Classes

In [ ]:
class DataCollector:
    def __init__(self, str_bucket, str_dirname_output):
        self.str_bucket = str_bucket
        self.str_dirname_output = str_dirname_output
        self.s3_client = boto3.client('s3')
        self.df_data = None

    def download_from_kaggle(self):
        """Download PaySim dataset from Kaggle using kagglehub."""
        print("Downloading PaySim dataset from Kaggle...")
        str_path = kagglehub.dataset_download('ealaxi/paysim1')
        print(f"Dataset downloaded to: {str_path}")
        
        # Find the CSV file
        list_files = [f for f in os.listdir(str_path) if f.endswith('.csv')]
        if not list_files:
            raise FileNotFoundError("No CSV file found in downloaded dataset")
        
        str_csv_path = os.path.join(str_path, list_files[0])
        print(f"Loading CSV: {str_csv_path}")
        self.df_data = pd.read_csv(str_csv_path)
        return self.df_data

    def clean_data(self):
        """Perform basic data cleaning."""
        if self.df_data is None:
            raise ValueError("Data not loaded. Call download_from_kaggle() first.")
        
        print("\nCleaning data...")
        
        # Check for nulls
        int_nulls = self.df_data.isnull().sum().sum()
        print(f"Total null values: {int_nulls}")
        
        # Remove any rows with nulls
        self.df_data = self.df_data.dropna()
        
        # Ensure correct dtypes
        self.df_data['step'] = self.df_data['step'].astype('int64')
        self.df_data['amount'] = self.df_data['amount'].astype('float64')
        self.df_data['oldbalanceOrg'] = self.df_data['oldbalanceOrg'].astype('float64')
        self.df_data['newbalanceOrig'] = self.df_data['newbalanceOrig'].astype('float64')
        self.df_data['oldbalanceDest'] = self.df_data['oldbalanceDest'].astype('float64')
        self.df_data['newbalanceDest'] = self.df_data['newbalanceDest'].astype('float64')
        self.df_data['isFraud'] = self.df_data['isFraud'].astype('int64')
        self.df_data['isFlaggedFraud'] = self.df_data['isFlaggedFraud'].astype('int64')
        
        print(f"Data shape after cleaning: {self.df_data.shape}")
        return self.df_data

    def get_summary_stats(self):
        """Compute and print summary statistics."""
        if self.df_data is None:
            raise ValueError("Data not loaded.")
        
        print("\n=== SUMMARY STATISTICS ===")
        print(f"Total records: {len(self.df_data):,}")
        print(f"Total features: {len(self.df_data.columns)}")
        print(f"\nFraud statistics:")
        print(f"  Non-fraud: {(self.df_data['isFraud'] == 0).sum():,} ({(self.df_data['isFraud'] == 0).sum() / len(self.df_data) * 100:.2f}%)")
        print(f"  Fraud: {(self.df_data['isFraud'] == 1).sum():,} ({(self.df_data['isFraud'] == 1).sum() / len(self.df_data) * 100:.4f}%)")
        print(f"\nTransaction types:")
        print(self.df_data['type'].value_counts())
        print(f"\nAmount statistics:")
        print(self.df_data['amount'].describe())
        print(f"\nFeatures: {list(self.df_data.columns)}")

    def save_to_s3(self, str_filename):
        """Save cleaned CSV to S3."""
        if self.df_data is None:
            raise ValueError("Data not loaded.")
        
        print(f"\nSaving to S3: s3://{self.str_bucket}/00_data_collection/{str_filename}")
        
        # Save locally first
        str_local_path = os.path.join(self.str_dirname_output, str_filename)
        self.df_data.to_csv(str_local_path, index=False)
        
        # Upload to S3
        try:
            self.s3_client.upload_file(
                str_local_path,
                self.str_bucket,
                f'00_data_collection/{str_filename}'
            )
            print(f"Successfully uploaded {str_filename} to S3")
        except Exception as e:
            print(f"Error uploading to S3: {e}")
            print(f"File saved locally at: {str_local_path}")

## Constants

In [ ]:
str_bucket = 'anomaly-detection-demo'
str_task = 'data_collection'
str_dirname_output = './output'

## Output Directory

In [ ]:
try:
    os.mkdir(str_dirname_output)
except FileExistsError:
    pass
print(f"Output directory ready: {str_dirname_output}")

## Download and Prepare Data

In [ ]:
# Initialize collector
collector = DataCollector(str_bucket, str_dirname_output)

# Download from Kaggle
df_raw = collector.download_from_kaggle()

In [ ]:
# Display first few rows
print("First few rows of raw data:")
print(df_raw.head())
print(f"\nData shape: {df_raw.shape}")
print(f"\nColumn names and types:")
print(df_raw.dtypes)

In [ ]:
# Clean the data
df_cleaned = collector.clean_data()

In [ ]:
# Get summary statistics
collector.get_summary_stats()

In [ ]:
# Save to S3
collector.save_to_s3('paysim_clean.csv')

## Completion

In [ ]:
print("\n=== DATA COLLECTION COMPLETE ===")
print(f"File ready for next stage: s3://{str_bucket}/00_data_collection/paysim_clean.csv")